# Legal NER (460 documents)

Runs on the **same 460 documents that already have embeddings**, reading the **preprocessed** JSONs.

| Step | Where it runs | Output |
|---|---|---|
| 1. Config + schema check | main Colab env | - |
| 2. Python 3.10 venv with spaCy 3.2 + `en_legal_ner_trf` | separate venv (the model pins old spaCy) | - |
| 3. NER (block-level, with regex citation layer) | venv | `ner_v2/doc_XXXXX.json` |
| 4. NER quality report | main env | counts / coverage |

**Runtime -> Change runtime type -> T4 GPU** is recommended (NER on CPU works but is slow). Everything is saved to Drive and **resumable**: if Colab disconnects, re-run the same cells.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

BASE      = "/content/drive/MyDrive/inabs_project"      # your project folder
PRE_DIR   = f"{BASE}/processed"                       # processed/{train,test,validate}/*.json
EMB_DIR   = f"{BASE}/embeddings"                   # embeddings/{train,test,validate}/*.pt  (the 460 docs)
NER_DIR   = f"{BASE}/ner_v2"                       # output of step 3

Mounted at /content/drive


In [2]:
# Schema check: confirms the 460 docs (train/test/validate) and shows what the files look like
import json, torch
from pathlib import Path
from collections import Counter

emb_files = sorted(Path(EMB_DIR).rglob("*.pt"))
print("embedding files (= documents to process):", len(emb_files))
print("per split:", dict(Counter(p.parent.name for p in emb_files)))
assert emb_files, "no .pt files found - check EMB_DIR"

e0 = emb_files[0]
print("\nexample embedding file:", e0.relative_to(EMB_DIR))
pre_files = sorted(Path(PRE_DIR).rglob("*.json"))
print("preprocessed json files (more than 460 is fine):", len(pre_files), "| example:", pre_files[0].relative_to(PRE_DIR))

cand = Path(PRE_DIR) / e0.parent.name / f"{e0.stem}.json"
print("matching preprocessed file exists:", cand.exists(), "->", cand.name)
d = json.loads((cand if cand.exists() else pre_files[0]).read_text())
print("top-level keys:", list(d)[:12])
if isinstance(d.get("sections"), dict):
    print("sections:", {k: len(v) for k, v in d["sections"].items()})
elif isinstance(d.get("sentences"), list):
    print("sentences:", len(d["sentences"]), "| first entry:", str(d["sentences"][0])[:150])

e = torch.load(e0, map_location="cpu", weights_only=False)
print("\nembedding file keys:", list(e) if isinstance(e, dict) else type(e))
print("embedding shape:", (e["embeddings"] if isinstance(e, dict) else e).shape)

embedding files (= documents to process): 460
per split: {'test': 100, 'train': 300, 'validate': 60}

example embedding file: test/test_1038.pt
preprocessed json files (more than 460 is fine): 7128 | example: test/test_100.json
matching preprocessed file exists: True -> test_1038.json
top-level keys: ['doc_id', 'split', 'filename', 'status', 'stats', 'sentences', 'has_summary']
sentences: 43 | first entry: {'id': 0, 'text': 'TEST - Judgment 1038 Appeals Nos. 490 and 491 of 1958.', 'section': 'PREAMBLE', 'n_tokens': 11}

embedding file keys: ['doc_id', 'split', 'filename', 'num_sentences', 'embeddings', 'sent_ids', 'sections', 'num_tokens', 'num_chunks', 'model']
embedding shape: torch.Size([43, 768])


## Step 2: Python 3.10 environment for the legal NER model
The model needs `spacy<3.3` and `spacy-transformers<1.2`, which do not install on Colab's default Python. We build an isolated 3.10 venv with `uv` (takes a few minutes: it downloads PyTorch).

In [3]:
!pip -q install uv
!uv venv --python 3.10 --seed /content/nerenv
!uv pip install --python /content/nerenv/bin/python "numpy<2" "spacy>=3.2.2,<3.3.0" "spacy-transformers>=1.1.4,<1.2.0" "click<8.1" rapidfuzz tqdm

# uv rejects the official wheel file name ("...-any-py3-none-any.whl"), so save it under a valid version name.
!wget -q -O /content/en_legal_ner_trf-3.2.0-py3-none-any.whl https://huggingface.co/opennyaiorg/en_legal_ner_trf/resolve/main/en_legal_ner_trf-any-py3-none-any.whl
!uv pip install --python /content/nerenv/bin/python /content/en_legal_ner_trf-3.2.0-py3-none-any.whl || /content/nerenv/bin/python -m pip install https://huggingface.co/opennyaiorg/en_legal_ner_trf/resolve/main/en_legal_ner_trf-any-py3-none-any.whl

Using CPython 3.10.22
Creating virtual environment with seed packages at: nerenv
 + packaging==26.3
 + pip==26.3
 + setuptools==84.2.1
 + wheel==0.48.0
Activate with: source nerenv/bin/activate
Using Python 3.10.22 environment at: nerenv
Resolved 58 packages in 583ms
Prepared 56 packages in 25.20s
Installed 56 packages in 175ms
 + blis==0.7.11
 + catalogue==2.0.10
 + certifi==2026.11.20
 + charset-normalizer==3.5.2
 + click==8.0.4
 + cymem==2.0.16
 + filelock==4.1.0
 + fsspec==2026.9.0
 + hf-xet==1.8.0
 + huggingface-hub==0.36.2
 + idna==3.20
 + jinja2==3.1.6
 + langcodes==3.5.1
 + markupsafe==3.0.4
 + mpmath==1.3.0
 + murmurhash==1.0.16
 + networkx==3.4.2
 + numpy==1.26.4
 + nvidia-cublas-cu12==12.1.3.1
 + nvidia-cuda-cupti-cu12==12.1.105
 + nvidia-cuda-nvrtc-cu12==12.1.105
 + nvidia-cuda-runtime-cu12==12.1.105
 + nvidia-cudnn-cu12==8.9.2.26
 + nvidia-cufft-cu12==11.0.2.54
 + nvidia-curand-cu12==10.3.2.106
 + nvidia-cusolver-cu12==11.4.5.107
 + nvidia-cusparse-cu12==12.1.0.106
 + nvid

In [4]:
# OPTIONAL (GPU speed-up). If this fails or the model errors on GPU, skip it: the script falls back to CPU, or pass --cpu.
!uv pip install --python /content/nerenv/bin/python cupy-cuda12x

Using Python 3.10.22 environment at: nerenv
Resolved 3 packages in 143ms
Prepared 3 packages in 2.15s
Uninstalled 1 package in 12ms
Installed 3 packages in 42ms
 + cuda-pathfinder==1.8.3
 + cupy-cuda12x==14.2.0
 - numpy==1.26.4
 + numpy==2.2.6


In [5]:
# Sanity check of the model. The model card shows: Section 319 PROVISION | Cr.P.C. STATUTE | G. Sambiah RESPONDENT | 20th June 1984 DATE
!/content/nerenv/bin/python -c "import spacy; nlp=spacy.load('en_legal_ner_trf'); t='Section 319 Cr.P.C. contemplates a situation where the evidence adduced by the prosecution for Respondent No.3-G. Sambiah on 20th June 1984'; print([(e.text, e.label_) for e in nlp(t).ents])"


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<string>", line 1, in <module>
  File "/content/nerenv/lib/python3.10/site-packages/spacy/__init__.py", line 11, in <module>
    from thinc.api import prefer_gpu, require_gpu, require_cpu  # noqa: F401
  File "/content/nerenv/lib/python3.10/site-packages/thinc/api.py", line 2, in <module>
    from .initializers import normal_init, uniform_init, glorot_uniform_init, zero_init
  File "/content/nerenv/lib/python3.10/site-packages/thinc/initializers.py", line 4, in <module>
    from .backends import Ops
  File "/conten

## Step 3: NER script
Design choices:
* NER runs on **blocks of up to 25 consecutive sentences within one section** (PREAMBLE / JUDGMENT), then maps entity character offsets back to sentences. Party roles (PETITIONER vs RESPONDENT) need surrounding context, which isolated sentences do not give.
* A **regex layer** adds Section/Article/Rule, AIR/SCC citations and case numbers the model misses.
* Entity strings are **normalised and fuzzy-merged per document**, and an `entity_index` (entity -> sentence ids) is written for the graph builder.
* Sentence order = `PREAMBLE` then `JUDGMENT` (as in your Phase 2 JSON), so sentence i matches embedding row i.

In [6]:
%%writefile /content/run_ner.py
"""Legal NER over the preprocessed documents that already have embeddings.

Runs inside the Python 3.10 venv (spaCy 3.2 + en_legal_ner_trf). Writes one JSON per
document with entities per sentence plus an entity -> sentence-id index.
"""
import argparse, json, re, sys
from pathlib import Path
from collections import defaultdict

import spacy
from rapidfuzz import fuzz
from tqdm import tqdm

BLOCK = 25   # sentences per NER block (kept inside one section)

# labels that may create graph edges (DATE / LAWYER are noisy for linking sentences)
EDGE_LABELS = {"PETITIONER", "RESPONDENT", "JUDGE", "COURT", "STATUTE", "PROVISION",
               "PRECEDENT", "CASE_NUMBER", "ORG", "GPE", "OTHER_PERSON", "WITNESS", "CITATION"}
# labels whose surface forms may be fuzzy-merged inside one document
FUZZY_LABELS = {"PETITIONER", "RESPONDENT", "JUDGE", "COURT", "ORG", "GPE",
                "OTHER_PERSON", "WITNESS", "STATUTE", "PRECEDENT"}
PERSON_LABELS = {"JUDGE", "PETITIONER", "RESPONDENT", "WITNESS", "OTHER_PERSON", "LAWYER"}

REGEX = [
    ("PROVISION",   re.compile(r"\b(?i:sections?|sec\.|articles?|art\.)\s+\d+[A-Za-z]?(?:\(\d+\))?(?:\([a-z]\))?")),
    ("PROVISION",   re.compile(r"\b(?:Rules?|Order)\s+\d+[A-Za-z]?")),
    ("CITATION",    re.compile(r"\bAIR\s+\d{4}\s+[A-Za-z]{2,}\s+\d+")),
    ("CITATION",    re.compile(r"\(\d{4}\)\s+\d+\s+(?:SCC|SCR|SCALE)\s+\d+")),
    ("CITATION",    re.compile(r"\[\d{4}\]\s+(?:\d+\s+)?(?:SCR|SCC|AC|A\.C\.|KB|K\.B\.|QB|Q\.B\.|All ER|WLR)\s+\d+")),
    ("CASE_NUMBER", re.compile(r"\b(?:(?:Civil|Criminal|Writ|Special Leave)\s+)?(?:Appeals?|Petitions?|Cases?)\s+Nos?\.?\s*\d+[A-Za-z]?(?:\s*(?:of|/)\s*\d{4})?")),
]

TITLES = re.compile(r"\b(?:hon'?ble|mr|mrs|ms|dr|shri|sri|smt|justice|learned)\b\.?")
STATUTE_ALIAS = {"ipc": "indian penal code", "crpc": "code of criminal procedure",
                 "cpc": "code of civil procedure", "constitution": "constitution of india"}

nlp = None   # loaded in main() (kept global so it can be stubbed in tests)


def normalize(text, label):
    t = text.lower().strip()
    if label in PERSON_LABELS:
        t = TITLES.sub(" ", t)
    if label == "JUDGE":
        t = re.sub(r"\s+jj?\.?$", "", t.strip())          # "Fazl Ali J." -> "fazl ali"
    if label == "STATUTE":
        k = re.sub(r"[\s\.]", "", t)
        t = STATUTE_ALIAS.get(k, t)
    t = re.sub(r"[^\w\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    t = re.sub(r"^the\s+", "", t)
    return t


def merge_variants(ents, thresh=90):
    """Within one document map near-duplicate names to the longest surface form."""
    canon, by_label = {}, defaultdict(set)
    for e in ents:
        by_label[e["label"]].add(e["norm"])
    for label, names in by_label.items():
        reps = []
        for n in sorted(names, key=len, reverse=True):
            if label in FUZZY_LABELS:
                for r in reps:
                    if fuzz.token_set_ratio(n, r) >= thresh:
                        canon[(label, n)] = r
                        break
                else:
                    reps.append(n); canon[(label, n)] = n
            else:
                canon[(label, n)] = n
    for e in ents:
        e["key"] = canon[(e["label"], e["norm"])]


def load_sentences(d):
    """Return [(section, text), ...] in the SAME order the embeddings were produced."""
    def txt(x):
        if isinstance(x, str):
            return x.strip()
        for k in ("text", "original_text", "sentence"):
            if k in x:
                return str(x[k]).strip()
        raise ValueError(f"cannot find text in sentence entry: {list(x)[:5]}")
    out = []
    if isinstance(d.get("sentences"), list) and d["sentences"]:
        for s in d["sentences"]:
            sec = s.get("section", "UNK") if isinstance(s, dict) else "UNK"
            out.append((sec, txt(s)))
    elif isinstance(d.get("sections"), dict):
        for sec, lst in d["sections"].items():
            for s in lst:
                out.append((sec, txt(s)))
    else:
        raise ValueError(f"unknown preprocessed schema, keys = {list(d)[:10]}")
    return out


def make_blocks(sents):
    blocks, cur = [], []
    for i, (sec, text) in enumerate(sents):
        if cur and (len(cur) >= BLOCK or sec != cur[-1][1]):
            blocks.append(cur); cur = []
        cur.append((i, sec, text))
    if cur:
        blocks.append(cur)
    return blocks


def overlaps(a, b):
    return a[0] < b[1] and b[0] < a[1]


def process_doc(doc_id, sents):
    per_sent = {i: [] for i in range(len(sents))}
    texts, offsets, blocks = [], [], make_blocks(sents)
    for blk in blocks:
        pos, spans, parts = 0, [], []
        for i, sec, text in blk:
            parts.append(text)
            spans.append((i, pos, pos + len(text)))
            pos += len(text) + 1               # blocks are joined by one space
        texts.append(" ".join(parts)); offsets.append(spans)

    for bdoc, spans in zip(nlp.pipe(texts, batch_size=8), offsets):
        found = [(e.start_char, e.end_char, e.label_, e.text, "model") for e in bdoc.ents]
        taken = [(a, b) for a, b, *_ in found]
        for label, rx in REGEX:                # regex hits that do not overlap a model entity
            for m in rx.finditer(bdoc.text):
                if not any(overlaps((m.start(), m.end()), t) for t in taken):
                    found.append((m.start(), m.end(), label, m.group().strip(), "regex"))
                    taken.append((m.start(), m.end()))
        for a, b, label, txt, src in found:
            for sid, sa, sb in spans:
                if sa <= a < sb:               # sentence that contains the entity start
                    per_sent[sid].append({"text": txt, "label": label, "source": src,
                                          "start": a - sa, "end": min(b, sb) - sa,
                                          "norm": normalize(txt, label)})
                    break

    all_ents = [e for lst in per_sent.values() for e in lst]
    for e in all_ents:
        if not e["norm"]:
            e["norm"] = e["text"].lower()
    merge_variants(all_ents)

    out, index = [], defaultdict(set)
    for i, (sec, text) in enumerate(sents):
        ents = per_sent[i]
        for e in ents:
            if e["label"] in EDGE_LABELS and e["key"]:
                index[f'{e["label"]}::{e["key"]}'].add(i)
        out.append({"sent_id": i, "section": sec, "text": text, "entities": ents,
                    "entity_labels": sorted({e["label"] for e in ents})})
    return {"doc_id": doc_id, "num_sentences": len(sents), "ner_backend": "en_legal_ner_trf+regex",
            "sentences": out, "entity_index": {k: sorted(v) for k, v in index.items()}}


def list_jobs(emb_dir):
    """[(split, doc_id), ...] for every embedding file, searching split sub-folders (train/test/validate)."""
    emb_dir = Path(emb_dir)
    jobs = []
    for p in sorted(emb_dir.rglob("*.pt")):
        rel = p.parent.relative_to(emb_dir)
        jobs.append((str(rel) if str(rel) != "." else "", p.stem))
    return jobs


_index = {}


def find_preprocessed(pre_dir, split, doc_id):
    p = pre_dir / split / f"{doc_id}.json"
    if p.exists():
        return p
    if not _index:                              # fall back: index all jsons by (split, stem)
        for f in pre_dir.rglob("*.json"):
            rel = f.parent.relative_to(pre_dir)
            _index[(str(rel) if str(rel) != "." else "", f.stem)] = f
    return _index.get((split, doc_id))


def main():
    global nlp
    ap = argparse.ArgumentParser()
    ap.add_argument("--pre_dir", required=True)
    ap.add_argument("--emb_dir", required=True, help="only used to decide WHICH docs to process")
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--cpu", action="store_true")
    a = ap.parse_args()
    pre_dir, out_dir = Path(a.pre_dir), Path(a.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    jobs = list_jobs(a.emb_dir)
    if a.limit:
        jobs = jobs[:a.limit]
    print(f"{len(jobs)} documents selected (those with embeddings)")

    if not a.cpu:
        try:
            print("device:", "GPU" if spacy.prefer_gpu() else "CPU")
        except Exception as ex:
            print("GPU unavailable, using CPU:", ex)
    nlp = spacy.load("en_legal_ner_trf")

    failed = []
    for split, doc_id in tqdm(jobs):
        out = out_dir / split / f"{doc_id}.json"
        if out.exists():                        # resume support
            continue
        try:
            src = find_preprocessed(pre_dir, split, doc_id)
            if src is None:
                raise FileNotFoundError(f"no preprocessed json for {split}/{doc_id}")
            sents = load_sentences(json.loads(src.read_text()))
            out.parent.mkdir(parents=True, exist_ok=True)
            out.write_text(json.dumps(process_doc(doc_id, sents), ensure_ascii=False))
        except Exception as ex:
            failed.append((split, doc_id, repr(ex)))
            print("FAILED", split, doc_id, repr(ex), file=sys.stderr)
    if failed:
        (out_dir / "_failed.json").write_text(json.dumps(failed, indent=1))
    print(f"done. failed: {len(failed)}")


if __name__ == "__main__":
    main()


Writing /content/run_ner.py


In [13]:
# Clean rebuild of the NER environment (no cupy)
!rm -rf /content/nerenv
!uv venv --python 3.10 --seed /content/nerenv
!uv pip install --python /content/nerenv/bin/python "numpy<2" "spacy>=3.2.2,<3.3.0" "spacy-transformers>=1.1.4,<1.2.0" "click<8.1" rapidfuzz tqdm

# model wheel (re-download in case the runtime was reset)
!wget -q -O /content/en_legal_ner_trf-3.2.0-py3-none-any.whl https://huggingface.co/opennyaiorg/en_legal_ner_trf/resolve/main/en_legal_ner_trf-any-py3-none-any.whl
!uv pip install --python /content/nerenv/bin/python /content/en_legal_ner_trf-3.2.0-py3-none-any.whl

# Check: should print numpy 1.x, spacy 3.2.x and "model ok"
!/content/nerenv/bin/python -c "import numpy, spacy; print(numpy.__version__, spacy.__version__); nlp = spacy.load('en_legal_ner_trf'); print('model ok')"

Using CPython 3.10.22
Creating virtual environment with seed packages at: nerenv
 + packaging==26.3
 + pip==26.3
 + setuptools==84.2.1
 + wheel==0.48.0
Activate with: source nerenv/bin/activate
Using Python 3.10.22 environment at: nerenv
Resolved 58 packages in 21ms
Installed 56 packages in 171ms
 + blis==0.7.11
 + catalogue==2.0.10
 + certifi==2026.11.20
 + charset-normalizer==3.5.2
 + click==8.0.4
 + cymem==2.0.16
 + filelock==4.1.0
 + fsspec==2026.9.0
 + hf-xet==1.8.0
 + huggingface-hub==0.36.2
 + idna==3.20
 + jinja2==3.1.6
 + langcodes==3.5.1
 + markupsafe==3.0.4
 + mpmath==1.3.0
 + murmurhash==1.0.16
 + networkx==3.4.2
 + numpy==1.26.4
 + nvidia-cublas-cu12==12.1.3.1
 + nvidia-cuda-cupti-cu12==12.1.105
 + nvidia-cuda-nvrtc-cu12==12.1.105
 + nvidia-cuda-runtime-cu12==12.1.105
 + nvidia-cudnn-cu12==8.9.2.26
 + nvidia-cufft-cu12==11.0.2.54
 + nvidia-curand-cu12==10.3.2.106
 + nvidia-cusolver-cu12==11.4.5.107
 + nvidia-cusparse-cu12==12.1.0.106
 + nvidia-nccl-cu12==2.18.1
 + nvidia-n

In [15]:
PY = "/content/nerenv/bin/python"
# Smoke test on 3 documents first (outputs are kept; the full run below skips them)
!{PY} /content/run_ner.py --pre_dir {PRE_DIR} --emb_dir {EMB_DIR} --out_dir {NER_DIR} --limit 3

3 documents selected (those with embeddings)
device: CPU
/content/nerenv/lib/python3.10/site-packages/torch/__init__.py:614: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:451.)
  _C._set_default_tensor_type(t)

done. failed: 0


In [16]:
# Inspect the smoke-test output before spending time on all 460
import json
from pathlib import Path
f = sorted(p for p in Path(NER_DIR).rglob("*.json") if not p.name.startswith("_"))[0]
d = json.loads(f.read_text())
print("file:", f.relative_to(NER_DIR))
for s in d["sentences"][:12]:
    print(s["sent_id"], s["section"], "|", s["text"][:90])
    for e in s["entities"]:
        print("      ", e["label"], "->", e["text"], f"[{e['source']}]")
print("\nlinking entities in this doc:", len(d["entity_index"]))

file: test/test_1038.json
0 PREAMBLE | TEST - Judgment 1038 Appeals Nos. 490 and 491 of 1958.
       CASE_NUMBER -> 1038 Appeals Nos. 490 and 491 of 1958 [model]
1 PREAMBLE | Appeals from the judgment and decree dated February 18, 1955, of the Madras High Court in 
       DATE -> February 18, 1955 [model]
       COURT -> Madras High Court [model]
       CASE_NUMBER -> Second Appeals Nos. 2038 and 2039 of 1950 [model]
2 PREAMBLE | N. R. Raghavachariar, M. R. Krishnaswami and T. V. R. Tatachari, for the appellant.
       OTHER_PERSON -> N. R. Raghavachariar [model]
       OTHER_PERSON -> M. R. Krishnaswami [model]
       OTHER_PERSON -> T. V. R. Tatachari [model]
3 PREAMBLE | R. Ganapathi Iyer and D. Gupta, for the respondent.
       OTHER_PERSON -> R. Ganapathi Iyer [model]
       OTHER_PERSON -> D. Gupta [model]
4 PREAMBLE | The Judgment of the Court was delivered by KAPUR, J. Two suits were brought by the appella
       JUDGE -> KAPUR [model]
       ORG -> State of Madras [model]
5 PR

In [17]:
# FULL RUN: all documents that have embeddings (resumable)
!{PY} /content/run_ner.py --pre_dir {PRE_DIR} --emb_dir {EMB_DIR} --out_dir {NER_DIR}

460 documents selected (those with embeddings)
device: CPU
/content/nerenv/lib/python3.10/site-packages/torch/__init__.py:614: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:451.)
  _C._set_default_tensor_type(t)

done. failed: 0


## Step 4: NER quality report

In [18]:
import json
from pathlib import Path
from collections import Counter

files = sorted(p for p in Path(NER_DIR).rglob("*.json") if not p.name.startswith("_"))
lab, src, with_ent, total, per_doc = Counter(), Counter(), 0, 0, []
for f in files:
    d = json.loads(f.read_text())
    per_doc.append(len(d["entity_index"]))
    for s in d["sentences"]:
        total += 1
        with_ent += bool(s["entities"])
        for e in s["entities"]:
            lab[e["label"]] += 1; src[e["source"]] += 1

print(f"documents: {len(files)}   per split: {dict(Counter(f.parent.name for f in files))}   sentences: {total}")
print(f"sentences with >=1 entity: {with_ent/total:.1%}")
print("entity source:", dict(src))
print("label counts:", lab.most_common())
print(f"linking entities per doc: mean {sum(per_doc)/len(per_doc):.1f}, min {min(per_doc)}, max {max(per_doc)}")
fail = Path(NER_DIR) / "_failed.json"
print("failed docs:", json.loads(fail.read_text()) if fail.exists() else "none")

documents: 460   per split: {'test': 100, 'train': 300, 'validate': 60}   sentences: 42110
sentences with >=1 entity: 50.1%
entity source: {'model': 51877, 'regex': 598}
label counts: [('PROVISION', 13473), ('OTHER_PERSON', 10181), ('STATUTE', 5647), ('DATE', 5192), ('GPE', 4380), ('ORG', 3686), ('PRECEDENT', 2391), ('COURT', 1954), ('CASE_NUMBER', 1924), ('JUDGE', 1567), ('LAWYER', 1035), ('WITNESS', 390), ('PETITIONER', 336), ('RESPONDENT', 316), ('CITATION', 3)]
linking entities per doc: mean 48.9, min 7, max 658
failed docs: none


Spot-check about 10 sentences by hand (PETITIONER / RESPONDENT / JUDGE / COURT / STATUTE should all appear) and note the error types for your report.

Output: one JSON per document in `NER_DIR`, each with entities per sentence and an `entity_index` (entity -> sentence ids) for later graph building.